# KNN Regressor – Reusable Template

**Short name:** `KNN_Regress`  
Drop in any numeric table with a continuous target. Scale on the training fold only.

```
load → split → MinMaxScaler.fit(train) → KNeighborsRegressor(k, weights) → MAE / sweep k → simulate
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score

# --- edit ---
CSV = "data/knn_movies_ratings.csv"
FEATURE_COLS = None          # None = all numeric except TARGET
TARGET = "rating"
TEST_SIZE = 0.25
SEED = 7
K_GRID = range(1, 21)
WEIGHTS = "distance"         # or "uniform"
# ------------

df = pd.read_csv(CSV)
y = df[TARGET].to_numpy(float)
num = df.select_dtypes(include=[np.number]).drop(columns=[TARGET])
X = num.to_numpy(float) if FEATURE_COLS is None else df[FEATURE_COLS].to_numpy(float)

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
sc = MinMaxScaler()
Xtr_s, Xva_s = sc.fit_transform(Xtr), sc.transform(Xva)

maes = []
for k in K_GRID:
    pred = KNeighborsRegressor(n_neighbors=k, weights=WEIGHTS).fit(Xtr_s, ytr).predict(Xva_s)
    maes.append(mean_absolute_error(yva, pred))
best_k = list(K_GRID)[int(np.argmin(maes))]
best = KNeighborsRegressor(n_neighbors=best_k, weights=WEIGHTS).fit(Xtr_s, ytr)
print(f"n={len(df)} features={X.shape[1]} best_k={best_k} MAE={min(maes):.4f} "
      f"R2={r2_score(yva, best.predict(Xva_s)):.3f}")

plt.plot(list(K_GRID), maes)
plt.axvline(best_k, ls="--")
plt.xlabel("k"); plt.ylabel("valid MAE"); plt.title("KNN regressor sweep")
plt.show()
